# scRareBench | High-Level scVI | Dataset 4 — COVID-19 autoimmunity PBMC

> **Reproducibility**: Notebook sourced from the project execution archive, with outputs cleared and the scRareBench dependency pinned to the reviewed D3/D4 taxonomy commit `cfdb720f5488b9ceef9a225b35fc7558bcb0251c`. Method configuration, method seed and dataset preprocessing settings are preserved.

This notebook intentionally keeps the workflow high-level:

`load_dataset(4) → MethodSpec(scVI) → benchmark_method() → HTML / multi-seed bundle`

Dataset-specific preparation is owned by **scRareBench**, not by this notebook.

The package revision is pinned to `cfdb720f5488b9ceef9a225b35fc7558bcb0251c`, whose current Dataset 4 contract is:

- dataset: `covid19_autoimmunity_pbmc`
- 97,499 cells × 21,858 features
- biological label: `cell_type` (**evaluation only**)
- integration/evaluation batch: `donor_id`
- canonical raw counts: `layers["counts"]`, aligned by the package from `adata.raw.X`
- benchmark-ready registered rare-scenario metadata
- scIB-reference HVG mode: `evaluation_batch`

For **method-side scVI preprocessing**, Dataset 4 uses the audited donor-aware
Seurat-v3 4,000-HVG policy with `span=0.3`. Both global and donor-aware
4000-HVG selection passed the Dataset 4 audit; unlike Dataset 3, no global-HVG
numerical workaround is required.

Biological labels are never passed to scVI.

## 1. Install the pinned scRareBench revision and scVI

In [ ]:
import subprocess
import sys

SCRAREBENCH_REVISION = "cfdb720f5488b9ceef9a225b35fc7558bcb0251c"
REPO = (
    "git+https://github.com/amirhossein-alishahi/"
    f"scRareBench_.git@{SCRAREBENCH_REVISION}"
)

subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "-q",
    "--no-deps",
    REPO,
])

METHOD_DEPENDENCIES = ("scvi-tools==1.4.3",)

from scrarebench.runtime import setup_runtime, print_install_report

install_report = setup_runtime(
    extra_requirements=METHOD_DEPENDENCIES,
    extra_imports=("scvi",),
    quiet=False,
)
print_install_report(install_report)


## 2. Load Dataset 4 through the package

In [ ]:
import scrarebench
from scrarebench import dataset_info, load_dataset, scenario_table_from_adata

adata = load_dataset(4)
info = dataset_info(adata)

# Fail-fast validation only; package-owned preparation is not duplicated here.
expected = {
    "dataset_key": "covid19_autoimmunity_pbmc",
    "label_key": "cell_type",
    "batch_key": "donor_id",
    "count_layer": "counts",
    "scib_hvg_batch_mode": "evaluation_batch",
    "benchmark_ready": True,
}
for key, value in expected.items():
    if info.get(key) != value:
        raise RuntimeError(
            f"Unexpected Dataset 4 contract for {key!r}: "
            f"observed={info.get(key)!r}, expected={value!r}"
        )

if adata.shape != (97_499, 21_858):
    raise RuntimeError(
        f"Unexpected Dataset 4 shape: {adata.shape}; expected (97499, 21858)."
    )
if "counts" not in adata.layers:
    raise RuntimeError("Dataset 4 canonical raw-count layer 'counts' is missing.")
if not adata.obs_names.is_unique:
    raise RuntimeError("Dataset 4 cell barcodes are not unique.")
if not adata.var_names.is_unique:
    raise RuntimeError("Dataset 4 feature identifiers are not unique.")

scenario_table = scenario_table_from_adata(adata)
if scenario_table is None or scenario_table.empty:
    raise RuntimeError("Dataset 4 registered rare-scenario metadata is missing.")
if len(scenario_table) != 12:
    raise RuntimeError(
        f"Unexpected Dataset 4 scenario-table size: {len(scenario_table)}; "
        "expected 12 registered six-state rows."
    )

print("scRareBench:", scrarebench.__version__)
print(adata)
print("\nDataset contract:")
for key in (
    "dataset_key",
    "label_key",
    "batch_key",
    "count_layer",
    "scib_hvg_batch_mode",
    "benchmark_ready",
):
    print(f"  {key}: {info.get(key)}")

print("\nRegistered six-state scenario rows:", len(scenario_table))
print("Scenario counts:")
print(scenario_table["scenario"].value_counts().sort_index())


## 3. scVI configuration

In [ ]:
# Method seeds vary; benchmark/evaluation randomness stays fixed.
METHOD_SEEDS = [42, 123, 2026]   # use [42] for a first smoke run
BENCHMARK_SEED = 42

METHOD_CONFIG = {
    # Dataset 4 method-side HVG policy from the completed audit.
    "n_hvg": 4000,
    "hvg_span": 0.3,

    # scVI architecture/training.
    "n_hidden": 128,
    "n_latent": 30,
    "n_layers": 2,
    "dropout_rate": 0.10,
    "dispersion": "gene-batch",
    "gene_likelihood": "nb",
    "max_epochs": 200,
    "batch_size": 256,
}


## 4. User-owned scVI method adapter

This is the only method-specific section.

The runner selects **donor-aware 4000 Seurat-v3 HVGs** from package-provided
canonical counts, supplies only `donor_id` as the scVI batch covariate, trains
scVI, validates the latent/barcode contract, and returns `MethodOutput`.

The biological label is not read by the runner. scIB/reference preprocessing is
performed separately later by scRareBench.


In [ ]:
import numpy as np
import scanpy as sc
import scvi

from scrarebench import MethodOutput, dataset_info


def run_scvi(method_adata, seed, config):
    method_info = dataset_info(method_adata)
    batch_key = method_info["batch_key"]
    count_layer = method_info["count_layer"]

    if method_info.get("dataset_key") != "covid19_autoimmunity_pbmc":
        raise RuntimeError("This runner is restricted to scRareBench Dataset 4.")
    if batch_key != "donor_id":
        raise RuntimeError(
            f"Unexpected Dataset 4 batch key: {batch_key!r}; expected 'donor_id'."
        )
    if method_info.get("scib_hvg_batch_mode") != "evaluation_batch":
        raise RuntimeError(
            "Dataset 4 is expected to use evaluation-batch-aware scIB HVGs."
        )
    if not count_layer or count_layer not in method_adata.layers:
        raise KeyError(f"Validated raw-count layer {count_layer!r} is missing.")

    original_barcodes = method_adata.obs_names.astype(str).copy()

    # Method-owned preprocessing: audited donor-aware Seurat-v3 HVGs.
    sc.pp.highly_variable_genes(
        method_adata,
        layer=count_layer,
        flavor="seurat_v3",
        n_top_genes=min(int(config["n_hvg"]), method_adata.n_vars),
        batch_key=batch_key,
        span=float(config["hvg_span"]),
        subset=False,
        check_values=True,
    )

    if "highly_variable" not in method_adata.var.columns:
        raise RuntimeError("Seurat-v3 HVG selection did not return a mask.")

    hvg_mask = (
        method_adata.var["highly_variable"]
        .fillna(False)
        .to_numpy(dtype=bool)
    )
    expected_hvg = min(int(config["n_hvg"]), method_adata.n_vars)
    selected_hvg = int(hvg_mask.sum())
    if selected_hvg != expected_hvg:
        raise RuntimeError(
            f"Expected {expected_hvg} HVGs, selected {selected_hvg}."
        )

    method_adata = method_adata[:, hvg_mask].copy()

    subset_barcodes = method_adata.obs_names.astype(str)
    if not subset_barcodes.equals(original_barcodes):
        raise RuntimeError(
            "Cell identity/order changed during method-side HVG subsetting."
        )

    scvi.settings.seed = int(seed)
    scvi.model.SCVI.setup_anndata(
        method_adata,
        layer=count_layer,
        batch_key=batch_key,
    )

    model = scvi.model.SCVI(
        method_adata,
        n_hidden=int(config["n_hidden"]),
        n_latent=int(config["n_latent"]),
        n_layers=int(config["n_layers"]),
        dropout_rate=float(config["dropout_rate"]),
        dispersion=str(config["dispersion"]),
        gene_likelihood=str(config["gene_likelihood"]),
    )

    model.train(
        max_epochs=int(config["max_epochs"]),
        train_size=0.90,
        validation_size=0.10,
        batch_size=int(config["batch_size"]),
        early_stopping=True,
        early_stopping_patience=20,
        accelerator="auto",
        devices="auto",
    )

    latent = np.asarray(model.get_latent_representation(), dtype=np.float32)

    expected_shape = (method_adata.n_obs, int(config["n_latent"]))
    if latent.shape != expected_shape:
        raise RuntimeError(
            f"Unexpected scVI latent shape {latent.shape}; expected {expected_shape}."
        )
    if not np.isfinite(latent).all():
        raise RuntimeError("scVI latent contains NaN or infinite values.")

    output_barcodes = method_adata.obs_names.astype(str)
    if not output_barcodes.equals(original_barcodes):
        raise RuntimeError(
            "scVI output barcode order differs from the loaded Dataset 4 order."
        )

    return MethodOutput(
        latent=latent,
        barcodes=output_barcodes,
        representation_key="X_scvi",
    )


## 5. Run the high-level benchmark

In [ ]:
from scrarebench import MethodSpec, benchmark_method

method = MethodSpec(
    name="scVI",
    runner=run_scvi,
    config=METHOD_CONFIG,
    dependencies=METHOD_DEPENDENCIES,
)

result = benchmark_method(
    adata,
    method,
    seeds=METHOD_SEEDS,
    output_dir="/content/scrarebench_results/dataset4_scvi",
    benchmark_config={
        "random_state": BENCHMARK_SEED,
        "write_pdf_report": False,
        "include_cell_ids": False,
    },
    install_dependencies=False,
    finalize=True,
)


## 6. Inspect and download the final result

In [ ]:
from IPython.display import display

display(result.summary().round(5))

print("Seeds:", result.seeds)
print("Multi-seed HTML:", result.report_path)
print("Delivery archive:", result.archive_path)
print("Summary JSON:", result.summary_path)

for seed, run in result.runs.items():
    print(
        f"seed={seed} | alignment={run.alignment} | "
        f"representation={run.representation_key}"
    )
    if run.alignment.get("exact_order_match") is not True:
        raise RuntimeError(f"Seed {seed} did not preserve exact barcode order.")

if result.archive_path is not None:
    try:
        from google.colab import files
        files.download(str(result.archive_path))
    except ImportError:
        print("Not running in Colab; archive remains at:", result.archive_path)


## Runtime note

For the first Colab execution you can temporarily use:

```python
METHOD_SEEDS = [42]
```

After the smoke run succeeds, restore `[42, 123, 2026]` for the final benchmark.

This notebook has been **statically validated**, not runtime-validated against a
live Colab GPU execution.
